# Практика · Класичні ознаки

> Лекція: [lecture.html](lecture.html) · Домашнє: [homework.md](homework.md) · Тест: [quiz.html](quiz.html)

**Мережа не потрібна:** усі зображення ми малюємо самі, формулами. Жодного файлу
не завантажується, тому числа в тебе на екрані будуть точнісінько такі самі,
як у лекції.

Що зробимо:

1. згенеруємо синтетичне «фото телефона на столі» — сцену цієї теми;
2. напишемо власну гістограму й звіримо її з `cv2.calcHist` через `np.allclose`;
3. побачимо на числах, що гістограма викидає **все** просторове;
4. побудуємо просторову піраміду й поміряємо, чим за неї заплачено;
5. перевіримо, чому напрямок градієнта стійкіший за яскравість;
6. зробимо **власний HOG** із градієнтів — комірка за коміркою;
7. знайдемо кути й заміряємо SIFT та ORB: точки, байти, час;
8. зіставимо два види предмета з тестом Лоу й `RANSAC`;
9. проведемо **негативний дослід**: покажемо, як ознака гине від того,
   до чого її автор не готував;
10. подивимось, чого саме більше немає в OpenCV 5.

## 0 · Що нам знадобиться

Три бібліотеки: `numpy` для масивів, `cv2` (OpenCV) для операцій із зображеннями
й `matplotlib`, щоб дивитись на результат очима.

In [ ]:
import time

import numpy as np
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

print("numpy ", np.__version__)
print("opencv", cv2.__version__)

## 1 · Сцена: телефон на деревʼяному столі

Функція нижче будує «фото товару з оголошення» чистою арифметикою над масивом.
Читається згори вниз: стільниця з прожилками, тінь, корпус, екран, сітка іконок,
зелений індикатор, відблиск на склі й нарешті шум матриці.

Дві деталі, які тут не випадкові, а зроблені навмисно:

- **іконки різного розміру.** Якби вони були однакові, кожен їхній ріг виглядав би
  так само, як одинадцять інших, і зіставлення точок наприкінці зошита загубилось би
  серед копій. Це не хитрість, а справжня проблема класичних ознак на повторюваних
  візерунках — просто ми не хочемо, щоб вона зіпсувала решту дослідів;
- **шум детермінований.** Значення кожного пікселя однозначно визначається його
  номером, тому знімок побайтово однаковий у всіх, хто запустить цей зошит.

In [ ]:
SCENE_HEIGHT, SCENE_WIDTH = 320, 400


def rounded_gap(row_grid, col_grid, left, top, right, bottom, radius):
    """Квадрат відстані від пікселя до скругленого прямокутника.

    Усередині фігури дає 0, тому одна й та сама функція годиться і щоб
    залити фігуру, і щоб намалювати навколо неї мʼяку тінь.
    """
    gap_x = np.maximum(np.maximum(left + radius - col_grid,
                                  col_grid - (right - radius)), 0.0)
    gap_y = np.maximum(np.maximum(top + radius - row_grid,
                                  row_grid - (bottom - radius)), 0.0)
    return gap_x * gap_x + gap_y * gap_y


def sensor_noise(height, width):
    """Детермінований «шум матриці»: значення залежить лише від номера пікселя."""
    index = np.arange(height * width * 3, dtype=np.int64)
    value = (index + 1) * 16807 % 2147483647
    value = value ^ (value >> 13)
    value = value * 48271 % 2147483647
    value = value ^ (value >> 17)
    value = value * 16807 % 2147483647
    return (value % 11 - 5).reshape(height, width, 3).astype(np.float64)


# у кожної іконки свій розмір і своя яскравість
ICON_COLORS = [
    (232,  84,  72), ( 74, 160, 232), (246, 196,  70), ( 96, 206, 130),
    (198,  96, 226), (240, 140,  60), ( 90, 214, 214), (226,  92, 150),
    (130, 150, 240), (250, 226,  90), (110, 200,  96), (200, 108,  92),
]
ICON_SIZES = [
    (34, 26), (26, 34), (32, 22), (24, 32), (36, 24), (28, 34),
    (32, 34), (30, 22), (22, 30), (28, 24), (34, 32), (24, 28),
]


def make_ad_scene():
    """Синтетичне фото телефона: масив (320, 400, 3) типу uint8, порядок каналів RGB."""
    row, col = np.mgrid[0:SCENE_HEIGHT, 0:SCENE_WIDTH].astype(np.float64)
    image = np.zeros((SCENE_HEIGHT, SCENE_WIDTH, 3), dtype=np.float64)

    # стільниця: дерево з поздовжніми прожилками
    grain = (np.sin(col * 0.16 + np.sin(row * 0.035) * 3.0) * 7.0
             + np.sin(col * 0.51 + row * 0.02) * 4.0)
    image[:, :, 0] = 198.0 + grain - 22.0 * row / SCENE_HEIGHT
    image[:, :, 1] = 172.0 + grain - 20.0 * row / SCENE_HEIGHT
    image[:, :, 2] = 138.0 + grain - 16.0 * row / SCENE_HEIGHT

    # тінь під корпусом
    distance = np.sqrt(rounded_gap(row, col, 130, 56, 300, 250, 16.0)) - 16.0
    image *= (1.0 - 0.32 * np.clip((16.0 - distance) / 16.0, 0.0, 1.0))[:, :, None]

    # корпус телефона
    body = rounded_gap(row, col, 122, 44, 292, 238, 16.0) <= 16.0 ** 2
    sheen = 12.0 * (1.0 - row / SCENE_HEIGHT)
    for channel, base in enumerate((52.0, 58.0, 68.0)):
        image[:, :, channel] = np.where(body, base + sheen, image[:, :, channel])

    # екран із діагональним переходом
    screen = rounded_gap(row, col, 130, 54, 284, 228, 5.0) <= 5.0 ** 2
    ramp = (col - 130.0) / 154.0 * 0.5 + (row - 54.0) / 174.0 * 0.5
    for channel, (start, span) in enumerate(((22.0, 120.0), (44.0, 70.0), (150.0, -90.0))):
        image[:, :, channel] = np.where(screen, start + span * ramp, image[:, :, channel])

    # сітка іконок 3 × 4 — головне джерело кутів на сцені
    for number, color in enumerate(ICON_COLORS):
        grid_row, grid_col = number // 3, number % 3
        left = 140.0 + grid_col * 44.0
        top = 70.0 + grid_row * 42.0
        width, height = ICON_SIZES[number]
        icon = rounded_gap(row, col, left, top, left + width, top + height, 7.0) <= 7.0 ** 2
        for channel in range(3):
            image[:, :, channel] = np.where(icon, float(color[channel]), image[:, :, channel])

    # зелений індикатор у нижньому кутку екрана
    indicator = rounded_gap(row, col, 268, 196, 278, 222, 4.0) <= 4.0 ** 2
    for channel, value in enumerate((40.0, 200.0, 90.0)):
        image[:, :, channel] = np.where(indicator, value, image[:, :, channel])

    # відблиск на склі: світла смуга під кутом, тільки в межах екрана
    band = (col - 130.0) * 0.80 + (row - 54.0) * 0.55
    glare = np.clip(1.0 - np.abs(band - 96.0) / 34.0, 0.0, 1.0)
    image += (glare * glare * 70.0 * screen)[:, :, None]

    image += sensor_noise(SCENE_HEIGHT, SCENE_WIDTH)
    return np.clip(image, 0, 255).astype(np.uint8)


scene = make_ad_scene()
gray = cv2.cvtColor(scene[:, :, ::-1].copy(), cv2.COLOR_BGR2GRAY)

print("сцена  :", scene.shape, scene.dtype)
print("сірий  :", gray.shape, gray.dtype)
print("пікселів у сірому кадрі:", gray.size)

Подивимось на сцену очима. Праворуч — той самий кадр у відтінках сірого:
далі майже всі ознаки рахуються саме з нього, бо і градієнти, і кути живуть
у яскравості, а не в кольорі.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].imshow(scene)
axes[0].set_title("фото товару з оголошення")
axes[1].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("той самий кадр у сірому")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("це синтетичне зображення, згенероване формулами вище")

## 2 · Своя гістограма проти `cv2.calcHist`

Гістограма — це просто підрахунок: скільки в кадрі пікселів кожної яскравості.
У NumPy для цього є `np.bincount`, і жодної магії всередині `cv2.calcHist` немає.
Переконаємось у цьому через `np.allclose`.

Одне зауваження про типи. `cv2.calcHist` завжди повертає `float32` і форму
`(корзин, 1)`, тому перед порівнянням ми випрямляємо його в один вимір.

In [ ]:
def my_histogram(image, bins):
    """Гістограма яскравості на bins корзин, порахована руками."""
    # 256 рівнів яскравості ділимо на bins рівних корзин
    bin_index = image.ravel().astype(np.int32) * bins // 256
    return np.bincount(bin_index, minlength=bins).astype(np.float32)


for bins in (256, 32, 16):
    mine = my_histogram(gray, bins)
    library = cv2.calcHist([gray], [0], None, [bins], [0, 256]).ravel()
    assert np.allclose(mine, library), f"розрахунок розійшовся на {bins} корзинах!"
    print(f"✅ {bins:>3} корзин — збігається; сума = {int(mine.sum())} = всі пікселі кадру")

## 3 · Гістограма викидає все просторове

Тепер найкоротший доказ, чому гістограма — слабка ознака. Візьмемо чотири
перетворення, які **переставляють** пікселі, не міняючи їхніх значень:

- перевернути кадр згори вниз;
- перевернути зліва направо;
- зсунути по колу на 17 рядків і 29 стовпців;
- перемішати пікселі повністю, у випадковому порядку.

Гістограма рахує тільки, скільки пікселів якого відтінку. Перестановка
кількість не міняє — отже, гістограма мусить збігтися **побітово**.
Перевіримо не «майже», а через `np.array_equal`.

In [ ]:
rng = np.random.default_rng(42)
shuffled = gray.ravel().copy()
rng.shuffle(shuffled)

transformed = {
    "перевернуто згори вниз": np.flipud(gray).copy(),
    "перевернуто зліва направо": np.fliplr(gray).copy(),
    "зсунуто по колу на 17 і 29": np.roll(gray, (17, 29), axis=(0, 1)).copy(),
    "пікселі перемішано повністю": shuffled.reshape(gray.shape),
}

original_histogram = cv2.calcHist([gray], [0], None, [256], [0, 256])

for name, variant in transformed.items():
    histogram = cv2.calcHist([variant], [0], None, [256], [0, 256])
    identical = np.array_equal(original_histogram, histogram)
    print(f"{name:<30} гістограма побітово та сама: {identical}")

Останній рядок варто прочитати двічі. Кадр, у якому **всі** пікселі перемішані у
випадковому порядку, для гістограми не відрізняється від оригіналу взагалі.
Подивимось на це очима.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("оригінал")
axes[1].imshow(transformed["пікселі перемішано повністю"], cmap="gray", vmin=0, vmax=255)
axes[1].set_title("пікселі перемішано")
axes[2].plot(original_histogram.ravel(), linewidth=1.2, label="оригінал")
axes[2].plot(cv2.calcHist([transformed["пікселі перемішано повністю"]],
                          [0], None, [256], [0, 256]).ravel(),
             linewidth=1.2, linestyle="--", label="перемішаний кадр")
axes[2].set_title("дві гістограми, накладені одна на одну")
axes[2].set_xlabel("яскравість")
axes[2].legend()
for axis in axes[:2]:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("дві криві праворуч збігаються повністю — тому видно лише одну")

### Гістограма кольору й `compareHist`

Гістограму яскравості ламає освітлення: притемнили кадр — усі стовпчики поїхали
ліворуч. Тому в реальних системах будують гістограму по каналах **H** (тон)
і **S** (насиченість) простору HSV, викидаючи канал **V**, який і є яскравістю.

Порівнюють такі гістограми функцією `cv2.compareHist`. Візьмемо метод
`HISTCMP_CORREL`: 1 — повний збіг, 0 — нічого спільного.

In [ ]:
def hue_saturation_histogram(rgb_image):
    """Гістограма тону й насиченості: 30 × 32 корзини, нормована."""
    hsv = cv2.cvtColor(rgb_image[:, :, ::-1].copy(), cv2.COLOR_BGR2HSV)
    histogram = cv2.calcHist([hsv], [0, 1], None, [30, 32], [0, 180, 0, 256])
    return cv2.normalize(histogram, histogram).flatten()


darker_scene = np.clip(scene.astype(np.float64) * 0.6, 0, 255).astype(np.uint8)
flipped_scene = np.flipud(scene).copy()

reference = hue_saturation_histogram(scene)
for name, variant in [("той самий кадр", scene),
                      ("перевернутий кадр", flipped_scene),
                      ("притемнений кадр ×0.6", darker_scene)]:
    similarity = cv2.compareHist(reference, hue_saturation_histogram(variant),
                                 cv2.HISTCMP_CORREL)
    print(f"{name:<24} близькість {similarity:6.3f}")

Тон майже не залежить від яскравості — ось чому притемнений кадр лишається
близьким. Але **перевернутий кадр теж лишається близьким**, і в цьому вся
слабкість: ознака не бачить, де саме що лежало.

## 4 · Просторова піраміда: повертаємо трохи простору

Перше компромісне рішення. Розріжемо кадр сіткою `n × n` і візьмемо гістограму
окремо в кожній клітинці, а потім склеїмо їх в один довгий вектор.

Міряти будемо двома числами:

- **стійкість** — близькість до кадру, зсунутого на 12 пікселів (має лишатись високою);
- **розрізняльність** — близькість до перевернутого кадру (має **падати**, бо
  перевернутий кадр — це інша картинка, і ознака мусить це побачити).

In [ ]:
def spatial_pyramid(image, cells_per_side, bins=16):
    """Гістограми в кожній клітинці сітки, склеєні в один вектор."""
    row_edges = np.linspace(0, image.shape[0], cells_per_side + 1).astype(int)
    col_edges = np.linspace(0, image.shape[1], cells_per_side + 1).astype(int)
    parts = []
    for row in range(cells_per_side):
        for col in range(cells_per_side):
            patch = image[row_edges[row]:row_edges[row + 1],
                          col_edges[col]:col_edges[col + 1]]
            histogram = my_histogram(patch, bins)
            parts.append(histogram / histogram.sum())     # частки, а не штуки
    return np.concatenate(parts)


def cosine_similarity(first, second):
    """1 — вектори дивляться в один бік, 0 — нічого спільного."""
    return float(first @ second / (np.linalg.norm(first) * np.linalg.norm(second)))


shifted = np.roll(gray, (0, 12), axis=(0, 1))
flipped = np.flipud(gray).copy()

print("клітинок  довжина   стійкість   розрізняльність")
print("               вектора   (зсув 12 px)  (переверт)")
for cells_per_side in (1, 2, 3, 4, 6, 8):
    base = spatial_pyramid(gray, cells_per_side)
    stability = cosine_similarity(base, spatial_pyramid(shifted, cells_per_side))
    flip_similarity = cosine_similarity(base, spatial_pyramid(flipped, cells_per_side))
    print(f"{cells_per_side} × {cells_per_side}      {len(base):>7}      {stability:6.3f}        {flip_similarity:6.3f}")

Прочитай таблицю двома колонками одразу. Довжина вектора виросла в десятки разів,
близькість до перевернутого кадру впала — ознака нарешті бачить різницю. Але й
стійкість до звичайного зсуву сповзла з рівної одиниці. **За розрізняльність
заплачено стійкістю**, і це той самий компроміс, який далі повториться в HOG і в SIFT.

## 5 · Градієнти стійкіші за яскравість

Тепер перевіримо головне твердження лекції: напрямок градієнта не залежить від
того, як яскраво світить лампа.

Логіка проста. Помножили всі пікселі на `k` — обидві складові градієнта
помножились на той самий `k`, а кут залежить лише від їхнього **відношення**,
яке не змінилось. Єдиний виняток — коли світлі пікселі впираються в стелю 255
і межа стає плоскою.

In [ ]:
def gradient(image):
    """Модуль і напрямок градієнта за Собелем. Напрямок у градусах, 0…180."""
    source = image.astype(np.float64)
    gradient_x = cv2.Sobel(source, cv2.CV_64F, 1, 0, ksize=3)
    gradient_y = cv2.Sobel(source, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.hypot(gradient_x, gradient_y)
    # напрямок беремо без знака: межа «темне-світле» і «світле-темне» — одна межа
    direction = np.rad2deg(np.arctan2(gradient_y, gradient_x)) % 180.0
    return magnitude, direction


base_magnitude, base_direction = gradient(gray)
strong = base_magnitude > 40           # рахуємо тільки там, де межа справді є

print("пікселів із помітною межею:", int(strong.sum()), "із", gray.size)
print()
print("множник  яскравість  модуль  вигоріло  відхилення напрямку")
for multiplier in (0.5, 0.8, 1.0, 1.2, 1.5, 1.8):
    brighter = np.clip(gray.astype(np.float64) * multiplier, 0, 255)
    magnitude, direction = gradient(brighter)
    gap = np.abs(direction - base_direction)
    gap = np.minimum(gap, 180 - gap)             # 179° і 1° — це різниця у 2°
    burnt = int((gray.astype(np.float64) * multiplier > 255).sum())
    print(f"  ×{multiplier:<5} {brighter.mean():9.2f} {magnitude[strong].mean():8.2f} "
          f"{burnt:9d} {gap[strong].mean():15.3f}°")

Три висновки з таблиці, і кожен точний:

1. поки нічого не вигоряє, **відхилення напрямку рівно `0.000°`** — не «майже нуль»,
   а нуль до останнього знака;
2. модуль градієнта множиться рівно на той самий коефіцієнт, що й яскравість;
3. щойно пікселі впираються в 255, напрямок починає пливти — і що більше вигоріло,
   то сильніше.

Ось звідки правило: **напрямок градієнта — найдешевша ознака, стійка до освітлення**.

## 6 · Власний HOG, крок за кроком

Тепер зберемо з напрямків справжню ознаку. HOG будується в чотири кроки, і ми
пройдемо кожен окремо, надрукувавши, що саме вийшло.

Вікно беремо класичне — **64 × 128 пікселів**, як у детекторі пішоходів, з якого
HOG і почався. Комірка 8 × 8, корзин 9, блок 2 × 2 комірки.

In [ ]:
CELL_SIZE = 8          # сторона комірки в пікселях
ORIENTATION_BINS = 9   # корзин напрямку: 0-20°, 20-40°, …, 160-180°

# вирізаємо телефон і зводимо його до класичного вікна детектора
phone_window = cv2.resize(gray[40:296, 112:304], (64, 128), interpolation=cv2.INTER_AREA)
print("вікно:", phone_window.shape, "→", phone_window.size, "чисел яскравості")


def cell_histograms(magnitude, direction, cell=CELL_SIZE, bins=ORIENTATION_BINS):
    """Крок 1-3: комірки, гістограма напрямків, голос вагою модуля."""
    rows, cols = magnitude.shape[0] // cell, magnitude.shape[1] // cell
    result = np.zeros((rows, cols, bins))
    # у яку корзину потрапляє напрямок кожного пікселя
    bin_of_pixel = np.minimum((direction / (180.0 / bins)).astype(int), bins - 1)
    for row in range(rows):
        for col in range(cols):
            piece_bins = bin_of_pixel[row * cell:(row + 1) * cell,
                                      col * cell:(col + 1) * cell].ravel()
            piece_weights = magnitude[row * cell:(row + 1) * cell,
                                      col * cell:(col + 1) * cell].ravel()
            # голос пікселя важить його модулем: різка межа важлива, шум — ні
            result[row, col] = np.bincount(piece_bins, weights=piece_weights, minlength=bins)
    return result


window_magnitude, window_direction = gradient(phone_window)
cells = cell_histograms(window_magnitude, window_direction)

print("крок 1-3: комірок", cells.shape[0], "×", cells.shape[1],
      "по", cells.shape[2], "корзин →", cells.size, "чисел")

Крок 4 — нормування по блоках. Беремо квадрат 2 × 2 сусідні комірки (це 4 × 9 = 36
чисел) і ділимо всі 36 на довжину цього вектора, тобто на корінь із суми квадратів.
Після цього в блоці важать не абсолютні величини, а **співвідношення** напрямків —
і саме тому HOG переживає зміну освітлення.

Блоки беруть із перекриттям: кожна комірка потрапляє в кілька блоків і щоразу
нормується по-різному.

In [ ]:
def block_normalise(cells, epsilon=1e-6):
    """Крок 4: блок 2 × 2 комірки ділиться на власну довжину."""
    rows, cols, bins = cells.shape
    blocks = []
    for row in range(rows - 1):          # -1, бо блоки перекриваються
        for col in range(cols - 1):
            block = cells[row:row + 2, col:col + 2].ravel()
            # epsilon рятує від ділення на нуль там, де межі немає взагалі
            blocks.append(block / np.sqrt((block * block).sum() + epsilon * epsilon))
    return np.concatenate(blocks)


def my_hog(window):
    """Повний HOG вікна: градієнти → комірки → нормування по блоках."""
    magnitude, direction = gradient(window)
    return block_normalise(cell_histograms(magnitude, direction))


hog_vector = my_hog(phone_window)
blocks_in_window = (cells.shape[0] - 1) * (cells.shape[1] - 1)

print("блоків у вікні :", blocks_in_window, f"({cells.shape[0] - 1} × {cells.shape[1] - 1})")
print("чисел у блоці  :", 2 * 2 * ORIENTATION_BINS)
print("довжина вектора:", len(hog_vector), "=", blocks_in_window, "×", 2 * 2 * ORIENTATION_BINS)
print()
print("це те саме число 3780, яке дає класичний HOG для вікна 64 × 128:",
      len(hog_vector) == 3780)
print()
print("перші 9 чисел вектора (перша комірка першого блоку):")
print(np.round(hog_vector[:9], 4))

Порівняти з бібліотечним HOG тут не вийде, і причина сама по собі є відповіддю на
питання теми: **`cv2.HOGDescriptor` в OpenCV 5 більше немає**. До цього ми
повернемось у розділі 11 останнім дослідом зошита.

Подивимось на вектор очима. Класична картинка HOG — «зірочка» в кожній комірці:
девʼять променів під різними кутами, довжина променя — сила відповідної корзини.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(8, 6.4))
axes[0].imshow(phone_window, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("вікно 64 × 128")
axes[0].axis("off")

axes[1].imshow(phone_window, cmap="gray", vmin=0, vmax=255, alpha=0.35)
strongest = cells.max()
for row in range(cells.shape[0]):
    for col in range(cells.shape[1]):
        center_x = col * CELL_SIZE + CELL_SIZE / 2
        center_y = row * CELL_SIZE + CELL_SIZE / 2
        for number in range(ORIENTATION_BINS):
            weight = cells[row, col, number] / strongest
            if weight < 0.03:
                continue
            angle = np.deg2rad((number + 0.5) * 180.0 / ORIENTATION_BINS)
            length = weight * CELL_SIZE / 2
            axes[1].plot([center_x - np.cos(angle) * length, center_x + np.cos(angle) * length],
                         [center_y - np.sin(angle) * length, center_y + np.sin(angle) * length],
                         color="deepskyblue", linewidth=1.1)
axes[1].set_title("гістограми напрямків у комірках")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("промені збігаються з межами іконок — саме їх HOG і бачить")

## 7 · Кутові точки

HOG описує ціле вікно одним вектором. Для іншої задачі — «де на другому знімку та
сама точка» — потрібні окремі **точки**, і не будь-які. Кут інформативніший за
межу, бо зсув віконця в будь-який бік змінює його вміст.

Поміряємо це прямо: візьмемо три місця й порахуємо, наскільки змінюється вміст
віконця 9 × 9 при зсуві на 3 пікселі в двадцяти чотирьох напрямках. Вирішує
**найменша** зміна, а не найбільша.

In [ ]:
def window_change(image, center_row, center_col, shift_row, shift_col, half=4):
    """Середній квадрат різниці між віконцем і ним самим, зсунутим на (shift_row, shift_col)."""
    first = image[center_row - half:center_row + half + 1,
                  center_col - half:center_col + half + 1].astype(np.float64)
    second = image[center_row - half + shift_row:center_row + half + 1 + shift_row,
                   center_col - half + shift_col:center_col + half + 1 + shift_col].astype(np.float64)
    return float(((first - second) ** 2).mean())


# три місця, вибрані навмисно: середина великої іконки, середина її прямого
# краю далеко від рогів і найсильніший кут сцени за відгуком Гарріса
spots = {"рівна пляма (всередині іконки)": (212, 201),
         "межа (край іконки)": (212, 184),
         "кут (ріг іконки)": (216, 167)}

harris_response = cv2.cornerHarris(np.float32(gray), 3, 3, 0.04)

print("місце                             найменша зміна  найбільша  відгук Гарріса")
for name, (row, col) in spots.items():
    changes = []
    for step in range(24):
        angle = step * np.pi / 12
        changes.append(window_change(gray, row, col,
                                     int(round(3 * np.sin(angle))),
                                     int(round(3 * np.cos(angle)))))
    print(f"{name:<34} {min(changes):11.1f} {max(changes):11.1f} "
          f"{harris_response[row, col]:15.3e}")

Читати таблицю треба обома числовими колонками одразу.

- У **плями** обидва числа малі: куди не зсунь віконце, у ньому те саме. Знайти цю
  точку на другому знімку неможливо — таких точок там тисячі.
- У **межі** найбільша зміна велика, а найменша — така сама мала, як у плями:
  упоперек лінії міняється все, вздовж лінії не міняється нічого.
- У **кута** велика вже й найменша зміна: у будь-який бік вміст віконця їде.

Саме найменшу зміну Гарріс і перетворює на одне число відгуку — у кута воно
додатне й велике, у межі різко відʼємне, у плями близьке до нуля. На практиці рідко викликають `cornerHarris` напряму:
є готова обгортка `cv2.goodFeaturesToTrack`, яка рахує відгук, відкидає слабкі
точки й розганяє решту, щоб вони не тіснились.

In [ ]:
corners = cv2.goodFeaturesToTrack(gray, maxCorners=300, qualityLevel=0.01, minDistance=8)
print("знайдено кутових точок:", len(corners))

plt.figure(figsize=(5.6, 4.4))
plt.imshow(gray, cmap="gray", vmin=0, vmax=255)
plt.scatter(corners[:, 0, 0], corners[:, 0, 1], s=14, c="orange", edgecolors="black",
            linewidths=0.4)
plt.title(f"{len(corners)} кутових точок")
plt.axis("off")
plt.show()

print("точки збіглися на рогах іконок і на межах корпусу — там, де сходяться дві межі")

## 8 · SIFT і ORB: точки, байти, час

Кут знайдено — лишається описати його околицю коротким вектором. Це роблять
дескриптори. В OpenCV 5 їх лишилось два: **SIFT** (128 чисел `float32`) і
**ORB** (32 байти `uint8`).

Заміряємо три речі: скільки точок знаходить кожен, скільки місця займає один
дескриптор і скільки часу йде на весь кадр. Час беремо як **мінімум** із кількох
прогонів — машина під навантаженням дає випадкові сплески, і мінімум чесніший
за середнє.

In [ ]:
sift = cv2.SIFT_create()
orb = cv2.ORB_create(nfeatures=500)

print("детектор  точок  байтів/точку  тип       час, мс  усього байтів")
measurements = {}
for name, detector in (("SIFT", sift), ("ORB", orb)):
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    timings = []
    for _ in range(7):
        start = time.perf_counter()
        detector.detectAndCompute(gray, None)
        timings.append(time.perf_counter() - start)
    bytes_per_point = descriptors.itemsize * descriptors.shape[1]
    measurements[name] = (len(keypoints), bytes_per_point, min(timings) * 1000)
    print(f"{name:<9} {len(keypoints):>5} {bytes_per_point:>13} {str(descriptors.dtype):<9} "
          f"{min(timings) * 1000:>7.1f} {descriptors.nbytes:>14}")

print()
print(f"ORB знайшов точок у {measurements['ORB'][0] / measurements['SIFT'][0]:.0f} разів більше,")
print(f"витратив на кожну в {measurements['SIFT'][1] // measurements['ORB'][1]} разів менше памʼяті")
print(f"і впорався у {measurements['SIFT'][2] / measurements['ORB'][2]:.0f} разів швидше")
print("(час у тебе буде свій — важливе тут співвідношення, а не абсолютна величина)")

## 9 · Зіставлення двох видів предмета

Ось задача, заради якої SIFT і ORB живі досі. Зробимо другий «знімок» тієї самої
сцени: повернемо її на 27°, зменшимо до 0.75 і трохи притемнимо.

Головне тут — **ми знаємо правильну відповідь наперед**. Перетворення задане
матрицею, тому для кожної пари можна порахувати, куди точка *мусила* переїхати,
і назвати пару правильною, лише якщо вона потрапила туди з точністю до 4 пікселів.
Без такої перевірки «кількість пар» нічого не означає.

In [ ]:
ROTATION_DEGREES, SCALE = 27.0, 0.75

transform = cv2.getRotationMatrix2D((SCENE_WIDTH / 2, SCENE_HEIGHT / 2),
                                    ROTATION_DEGREES, SCALE)
second_view = cv2.warpAffine(gray, transform, (SCENE_WIDTH, SCENE_HEIGHT),
                             flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
second_view = np.clip(second_view.astype(np.float64) * 0.85 + 10, 0, 255).astype(np.uint8)

figure, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("знімок 1")
axes[1].imshow(second_view, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("знімок 2: поворот 27°, масштаб 0.75, темніше")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("предмет той самий, кадр інший — саме такий випадок буває в панорамі чи в SLAM")

Тепер сам ланцюжок. Він складається з трьох етапів, і кожен наступний викидає
частину пар:

1. **найближчий сусід** — для кожної точки першого знімка беремо найсхожішу точку
   другого. Пар багато, сміття теж;
2. **тест співвідношення Лоу** — беремо *двох* найближчих сусідів і приймаємо пару,
   лише якщо перший помітно ближчий за другого. Якщо два кандидати однаково близькі,
   точка невиразна, і краще її викинути;
3. **`RANSAC`** — усі правильні пари мусять узгоджуватись однією геометрією.
   Алгоритм шукає її навмання й відкидає все, що в неї не вкладається.

In [ ]:
def project_points(points, matrix):
    """Куди точки першого знімка мусили переїхати за відомим перетворенням."""
    homogeneous = np.hstack([points, np.ones((len(points), 1))])
    return homogeneous @ matrix.T


def count_correct(points_first, points_second, matrix, tolerance=4.0):
    """Скільки пар справді відповідають одна одній — за геометрією, а не на око."""
    expected = project_points(points_first, matrix)
    error = np.linalg.norm(expected - points_second, axis=1)
    return int((error < tolerance).sum())


keypoints_first, descriptors_first = sift.detectAndCompute(gray, None)
keypoints_second, descriptors_second = sift.detectAndCompute(second_view, None)
matcher = cv2.BFMatcher(cv2.NORM_L2)
pairs = matcher.knnMatch(descriptors_first, descriptors_second, k=2)

# етап 1: просто найближчий сусід, без жодного фільтра
nearest = [first for first, _second in pairs]
points_a = np.float32([keypoints_first[m.queryIdx].pt for m in nearest])
points_b = np.float32([keypoints_second[m.trainIdx].pt for m in nearest])
correct_nearest = count_correct(points_a, points_b, transform)

# етап 2: тест Лоу з класичним порогом
LOWE_RATIO = 0.75
good = [first for first, second in pairs if first.distance < LOWE_RATIO * second.distance]
points_a = np.float32([keypoints_first[m.queryIdx].pt for m in good])
points_b = np.float32([keypoints_second[m.trainIdx].pt for m in good])
correct_good = count_correct(points_a, points_b, transform)

# етап 3: RANSAC шукає одну спільну геометрію
homography, mask = cv2.findHomography(points_a.reshape(-1, 1, 2),
                                      points_b.reshape(-1, 1, 2),
                                      cv2.RANSAC, 4.0)
inliers = mask.ravel().astype(bool)
correct_inliers = count_correct(points_a[inliers], points_b[inliers], transform)

print("етап                          пар  правильних  частка")
print(f"{'усі найближчі сусіди':<28} {len(nearest):>4} {correct_nearest:>11} "
      f"{100 * correct_nearest / len(nearest):>7.0f} %")
print(f"{'після тесту Лоу (0.75)':<28} {len(good):>4} {correct_good:>11} "
      f"{100 * correct_good / len(good):>7.0f} %")
print(f"{'після RANSAC':<28} {int(inliers.sum()):>4} {correct_inliers:>11} "
      f"{100 * correct_inliers / max(int(inliers.sum()), 1):>7.0f} %")

Подивимось, як поріг Лоу торгує повнотою за чистоту. Що вищий поріг, то більше
пар — і то менша частка правильних серед них.

In [ ]:
print("поріг Лоу  пар  правильних  частка")
for ratio in (0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1.0):
    selected = [first for first, second in pairs if first.distance < ratio * second.distance]
    first_points = np.float32([keypoints_first[m.queryIdx].pt for m in selected])
    second_points = np.float32([keypoints_second[m.trainIdx].pt for m in selected])
    correct = count_correct(first_points, second_points, transform)
    print(f"   {ratio:<7} {len(selected):>4} {correct:>11} {100 * correct / len(selected):>7.0f} %")

Намалюємо пари, що пройшли `RANSAC`. Зелена лінія — правильна пара,
червона — хибна.

In [ ]:
canvas = np.hstack([gray, second_view])
canvas = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)

expected_positions = project_points(points_a[inliers], transform)
actual_positions = points_b[inliers]
errors = np.linalg.norm(expected_positions - actual_positions, axis=1)

for start, end, error in zip(points_a[inliers], actual_positions, errors):
    color = (60, 180, 90) if error < 4.0 else (220, 60, 60)
    cv2.line(canvas,
             (int(start[0]), int(start[1])),
             (int(end[0]) + SCENE_WIDTH, int(end[1])),
             color, 1, cv2.LINE_AA)

plt.figure(figsize=(11, 4.4))
plt.imshow(canvas)
plt.title(f"{int(inliers.sum())} пар після RANSAC, із них правильних {correct_inliers}")
plt.axis("off")
plt.show()

print("усі лінії зелені — після RANSAC хибних пар не лишилось")

## 10 · Негативний дослід: ознака гине від того, до чого не готована

Досі все виглядало добре. Тепер чесний дослід, який показує межу підходу.

Візьмемо дві ознаки — гістограму яскравості й наш власний HOG — і прожену їх крізь
шість спотворень. Очікування просте й перевірюване: **гістограма мусить пережити
поворот і загинути від зміни світла, а HOG — навпаки**.

In [ ]:
def rotate(image, degrees):
    matrix = cv2.getRotationMatrix2D((image.shape[1] / 2, image.shape[0] / 2), degrees, 1.0)
    return cv2.warpAffine(image, matrix, (image.shape[1], image.shape[0]),
                          borderMode=cv2.BORDER_REPLICATE)


def rescale(image, factor):
    smaller = cv2.resize(image, None, fx=factor, fy=factor, interpolation=cv2.INTER_AREA)
    return cv2.resize(smaller, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)


def brighten(image, factor):
    return np.clip(image.astype(np.float64) * factor, 0, 255).astype(np.uint8)


def add_noise(image, sigma):
    noise_rng = np.random.default_rng(7)
    return np.clip(image.astype(np.float64) + noise_rng.normal(0, sigma, image.shape),
                   0, 255).astype(np.uint8)


distortions = {
    "поворот 20°": lambda window: rotate(window, 20),
    "поворот 45°": lambda window: rotate(window, 45),
    "масштаб 0.6": lambda window: rescale(window, 0.6),
    "яскравість ×0.6": lambda window: brighten(window, 0.6),
    "яскравість ×1.6": lambda window: brighten(window, 1.6),
    "шум σ = 20": lambda window: add_noise(window, 20),
}


def intensity_feature(window):
    histogram = my_histogram(window, 32)
    return histogram / histogram.sum()


reference_histogram = intensity_feature(phone_window)
reference_hog = my_hog(phone_window)

print("спотворення        гістограма   HOG")
for name, apply in distortions.items():
    distorted = apply(phone_window)
    print(f"{name:<18} {cosine_similarity(reference_histogram, intensity_feature(distorted)):9.3f} "
          f"{cosine_similarity(reference_hog, my_hog(distorted)):8.3f}")

Дві колонки майже дзеркальні, і це не збіг. Гістограма викинула координати —
поворот її не турбує, а зміна яскравості вбиває. HOG викинув абсолютну
яскравість — світло його не турбує, а поворот вбиває.

**Кожна придумана ознака стійка рівно до того, що її автор вирішив викинути, і
сліпа рівно до того ж.** Жоден поріг не зробить із цих двох колонок одну хорошу.

### Другий негативний дослід: екземпляр проти класу

Ще гостріше та сама межа видно, якщо змінити задачу. Досі ми питали «чи це той
самий предмет». Тепер спитаємо «чи це той самий **клас** предметів»: намалюємо
інший телефон — вужчий корпус, світлий чохол, темна тема на екрані, інша сітка
іконок. Людина скаже «телефон» за чверть секунди.

In [ ]:
DARK_PHONE_COLORS = [
    ( 96, 118, 150), ( 88, 108, 138), (104, 126, 158), ( 80, 100, 130),
    (110, 132, 164), ( 84, 104, 134), (100, 122, 152), ( 92, 112, 142),
    (106, 128, 160), ( 86, 106, 136), ( 98, 120, 150), ( 90, 110, 140),
]


def make_other_phone():
    """Той самий клас — інша модель: світлий чохол, темна тема, інша сітка іконок."""
    row, col = np.mgrid[0:SCENE_HEIGHT, 0:SCENE_WIDTH].astype(np.float64)
    image = np.zeros((SCENE_HEIGHT, SCENE_WIDTH, 3), dtype=np.float64)

    grain = (np.sin(row * 0.14 + np.sin(col * 0.03) * 3.0) * 6.0
             + np.sin(row * 0.44 + col * 0.02) * 4.0)
    image[:, :, 0] = 186.0 + grain - 18.0 * col / SCENE_WIDTH
    image[:, :, 1] = 178.0 + grain - 16.0 * col / SCENE_WIDTH
    image[:, :, 2] = 160.0 + grain - 14.0 * col / SCENE_WIDTH

    distance = np.sqrt(rounded_gap(row, col, 128, 70, 298, 236, 20.0)) - 20.0
    image *= (1.0 - 0.28 * np.clip((20.0 - distance) / 20.0, 0.0, 1.0))[:, :, None]

    body = rounded_gap(row, col, 118, 58, 290, 224, 20.0) <= 20.0 ** 2
    for channel, value in enumerate((214.0, 208.0, 196.0)):        # світлий чохол
        image[:, :, channel] = np.where(body, value, image[:, :, channel])

    screen = rounded_gap(row, col, 128, 70, 280, 212, 4.0) <= 4.0 ** 2
    for channel, value in enumerate((26.0, 30.0, 38.0)):           # темна тема
        image[:, :, channel] = np.where(screen, value, image[:, :, channel])

    for number, color in enumerate(DARK_PHONE_COLORS):
        grid_row, grid_col = number // 4, number % 4
        left = 82.0 + grid_col * 34.0
        top = 138.0 + grid_row * 50.0
        icon = rounded_gap(row, col, left, top, left + 24.0, top + 38.0, 5.0) <= 5.0 ** 2
        for channel in range(3):
            image[:, :, channel] = np.where(icon, float(color[channel]), image[:, :, channel])

    image += sensor_noise(SCENE_HEIGHT, SCENE_WIDTH)
    return np.clip(image, 0, 255).astype(np.uint8)


other_phone = make_other_phone()
other_gray = cv2.cvtColor(other_phone[:, :, ::-1].copy(), cv2.COLOR_BGR2GRAY)

figure, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].imshow(scene)
axes[0].set_title("телефон A")
axes[1].imshow(other_phone)
axes[1].set_title("телефон Б: той самий клас, інша модель")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("для людини це два телефони; подивимось, що скажуть ознаки")

In [ ]:
# скільки пар знаходить SIFT між РІЗНИМИ телефонами того самого класу
other_keypoints, other_descriptors = sift.detectAndCompute(other_gray, None)
class_pairs = matcher.knnMatch(descriptors_first, other_descriptors, k=2)
class_good = [first for first, second in class_pairs
              if first.distance < LOWE_RATIO * second.distance]

# і скільки — між тим САМИМ телефоном на двох знімках (це вже пораховано вище)
print("SIFT, той самий телефон під іншим кутом — пар:", len(good))
print("SIFT, інший телефон того самого класу   — пар:", len(class_good))
print()

# те саме HOG-ом: порівнюємо вікно телефона A з вікном телефона Б
other_window = cv2.resize(other_gray[52:300, 104:300], (64, 128), interpolation=cv2.INTER_AREA)
rotated_window = cv2.resize(rotate(gray, 20)[40:296, 112:304], (64, 128),
                            interpolation=cv2.INTER_AREA)

similarity_to_other_phone = cosine_similarity(reference_hog, my_hog(other_window))
similarity_to_own_rotated = cosine_similarity(reference_hog, my_hog(rotated_window))

print(f"HOG, телефон A проти телефона Б         : {similarity_to_other_phone:.3f}")
print(f"HOG, телефон A проти себе, поворот 20°  : {similarity_to_own_rotated:.3f}")
print()
if similarity_to_other_phone > similarity_to_own_rotated:
    print("⚠️ чужий телефон схожий БІЛЬШЕ, ніж власний, повернутий на 20°:")
    print("   жоден поріг не розділить ці два випадки")

Ось і вирок, і він у двох рядках.

**SIFT** упізнає **екземпляр**, а не клас: той самий апарат під іншим кутом дає
десятки узгоджених пар, інший апарат того ж класу — одиниці, тобто випадковість.
Для панорами це саме те, що треба; для питання «чи є на фото телефон» — не годиться
взагалі.

**HOG** узагалі помірявся не з тим: чужий телефон виявився ближчим за власний,
повернутий на двадцять градусів. Поставити поріг між цими двома числами неможливо.

## 11 · Чого більше немає в OpenCV 5

Останній дослід зошита — і найкоротший. Просто спитаємо бібліотеку, що в ній є.

In [ ]:
classic = ["HOGDescriptor", "CascadeClassifier", "AKAZE_create", "BRISK_create",
           "SIFT_create", "ORB_create"]

print("клас або функція        є в cv2 5.0?")
for name in classic:
    print(f"  {name:<22} {hasattr(cv2, name)}")

Два перші рядки ламають дві безсмертні демонстрації підручників: детектор
пішоходів на HOG і виявлення облич каскадами Хаара. Бібліотека не викидає класи з
примхи — вона викидає те, чим перестали користуватися.

Натомість в OpenCV 5 зʼявились `ALIKED` і `DISK` — детектори ознак, які не
придумані, а **навчені на даних**. Класи є, але створити їх без файлу моделі не
можна, а тягнути модель із мережі ми не будемо. Подивимось на це прямо: клітинка
нижче навмисно ловить виняток і друкує його текст.

In [ ]:
print("ALIKED є в cv2:", hasattr(cv2, "ALIKED"), "  DISK є в cv2:", hasattr(cv2, "DISK"))
print()
for name in ("ALIKED", "DISK"):
    try:
        getattr(cv2, name).create()
        print(f"{name}: створився без файлу моделі")
    except TypeError as error:
        # це очікувана поведінка, а не поломка: ваг у пакеті немає
        print(f"{name}: {error}")

print()
print("у тій самій бібліотеці, яка викинула HOG і каскади Хаара,")
print("місце придуманих детекторів зайняли навчені")

## Що далі

Ознаку тут ми щоразу **вигадували руками**: вирішували, що викинути й що лишити,
а потім перевіряли здогад числом. Останні два досліди показали, де цей шлях
упирається в стіну — ознака під один клас доводиться придумувати заново під інший.

Наступний блок починається з питання, чи не можна замість вигадування
**вивчити перетворення з даних**.

---

## Завдання

### 🟢 Рівень 1

Додай до таблиці розділу 10 третю колонку — гістограму тону й насиченості
(функція `hue_saturation_histogram` із розділу 3). Порівняй її з двома іншими
ознаками під тими самими шістьма спотвореннями й скажи словами, у чому вона
краща за гістограму яскравості й у чому — гірша за HOG.

### 🟡 Рівень 2

Поміряй ланцюжок із розділу 9 не для SIFT, а для ORB (відстань —
`cv2.NORM_HAMMING`). Побудуй ту саму таблицю з трьох етапів і порівняй числа
з SIFT: де ORB виграє, де програє й на якому порозі Лоу він перестає бути
надійним.

### 🔴 Рівень 3

Зроби HOG інваріантним до повороту. Ідея з розділу про SIFT: спершу знайди
переважний напрямок градієнта у вікні, а потім відлічуй усі напрямки від нього
(тобто циклічно зсунь корзини). Поміряй, наскільки виріс рядок «поворот 20°»
у таблиці розділу 10 — і чи не впала при цьому колонка «яскравість».